# Praktikum Machine Learning: Prediksi Biaya Asuransi

**Nama:** Billy Nanda Alfahrizi  
**Dataset:** Medical Cost Personal Dataset (`insurance.csv`)  
**Jenis masalah:** Regresi

Notebook ini menganalisis dataset Medical Cost Personal Dataset untuk memprediksi `charges` (biaya asuransi) dari usia, jenis kelamin, BMI, jumlah anak, status merokok, dan wilayah. Target berupa angka, sehingga ini merupakan masalah regresi.

## 1. Mengimpor library

Blok ini mengimpor pustaka untuk pengolahan data (`pandas`, `numpy`), visualisasi (`matplotlib`, `seaborn`), penyimpanan model (`joblib`), serta pembagian data, preprocessing, pelatihan, dan evaluasi model dari scikit-learn.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline

from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("Semua library berhasil diimpor.")

## 2. Memuat dataset

Kode membaca dataset dari file `insurance.csv` yang berada di folder kerja notebook. Pastikan nama file tersebut tetap `insurance.csv` dan berada di folder yang sama.

In [ ]:
df = pd.read_csv("insurance.csv")

print("Ukuran dataset:", df.shape)
display(df.head())


## 3. Memeriksa struktur dan kualitas data

Bagian ini menampilkan informasi tipe kolom, jumlah nilai kosong, jumlah baris duplikat, serta ringkasan statistik. Pemeriksaan awal membantu mengetahui kondisi data sebelum dibersihkan.

In [ ]:
print("Informasi dataset:")
df.info()

print("\nJumlah data kosong setiap kolom:")
display(df.isnull().sum())

print("\nJumlah baris duplikat:", df.duplicated().sum())

print("\nStatistik deskriptif:")
display(df.describe())

## 4. Membersihkan data

Blok ini menghapus baris duplikat, memastikan target `charges` tersedia, dan menangani nilai kosong pada kolom lain sesuai tipe datanya. Langkah ini membantu mencegah masalah saat preprocessing dan pelatihan.

In [ ]:
# Menghapus baris duplikat jika ada
df = df.drop_duplicates().copy()

# Memastikan kolom target tidak kosong
df = df.dropna(subset=["charges"])

# Mengisi data kategorikal yang kosong dengan modus
categorical_cols = ["sex", "smoker", "region"]

for col in categorical_cols:
    if df[col].isnull().any():
        df[col] = df[col].fillna(df[col].mode()[0])

# Mengisi data numerik yang kosong dengan median
numeric_cols = ["age", "bmi", "children"]

for col in numeric_cols:
    if df[col].isnull().any():
        df[col] = df[col].fillna(df[col].median())

print("Ukuran dataset setelah pembersihan:", df.shape)
print("\nJumlah data kosong:")
display(df.isnull().sum())

## 5. Memahami kolom dan skala pengukuran

Tabel ini merangkum nama variabel, tipe datanya, dan skala pengukuran. Informasi tersebut membantu menentukan kolom yang dipakai sebagai fitur, target, dan metode preprocessing yang sesuai.

In [ ]:
jenis_data = pd.DataFrame({
    "Variabel": [
        "age", "bmi", "children", "charges",
        "sex", "smoker", "region"
    ],
    "Jenis_data": [
        "Numerik - umumnya kontinu",
        "Numerik - kontinu",
        "Numerik - diskrit",
        "Numerik - kontinu",
        "Kategorikal",
        "Kategorikal",
        "Kategorikal"
    ],
    "Skala_pengukuran": [
        "Rasio", "Rasio", "Rasio", "Rasio",
        "Nominal", "Nominal", "Nominal"
    ],
    "Keterangan": [
        "Usia",
        "Indeks massa tubuh",
        "Jumlah anak",
        "Biaya asuransi (target)",
        "Kategori jenis kelamin",
        "Status merokok",
        "Wilayah tempat tinggal"
    ]
})

display(jenis_data)

## 6. Ringkasan statistik biaya asuransi

Kode menghitung rata-rata, median, nilai minimum, dan maksimum `charges`. Statistik ini memberikan gambaran awal mengenai besaran biaya asuransi dalam dataset.

In [ ]:
print("Rata-rata biaya asuransi:", round(df["charges"].mean(), 2))
print("Median biaya asuransi:", round(df["charges"].median(), 2))
print("Biaya asuransi minimum:", round(df["charges"].min(), 2))
print("Biaya asuransi maksimum:", round(df["charges"].max(), 2))

plt.figure(figsize=(8, 5))
sns.histplot(df["charges"], kde=True)
plt.title("Distribusi Biaya Asuransi")
plt.xlabel("Charges")
plt.ylabel("Frekuensi")
plt.show()

## 7. Hubungan usia dan biaya asuransi

Scatter plot ini memvisualisasikan hubungan antara usia dan biaya asuransi, dengan warna yang membedakan status merokok. Grafik membantu mengamati pola hubungan, tetapi tidak dengan sendirinya membuktikan sebab-akibat.

In [ ]:
plt.figure(figsize=(7, 5))
sns.scatterplot(data=df, x="age", y="charges", hue="smoker")
plt.title("Hubungan Usia dan Biaya Asuransi")
plt.xlabel("Usia")
plt.ylabel("Biaya Asuransi")
plt.show()

plt.figure(figsize=(7, 5))
sns.boxplot(data=df, x="smoker", y="charges")
plt.title("Perbandingan Biaya Asuransi Berdasarkan Status Merokok")
plt.xlabel("Status Merokok")
plt.ylabel("Biaya Asuransi")
plt.show()

## 8. Menentukan fitur (X) dan target (y)

`X` berisi kolom input yang digunakan model untuk membuat prediksi, sedangkan `y` berisi nilai yang akan diprediksi, yaitu `charges`. Pemisahan ini merupakan tahap dasar dalam supervised learning.

In [ ]:
X = df[["age", "sex", "bmi", "children", "smoker", "region"]]
y = df["charges"]

print("Variabel input:")
display(X.head())

print("\nVariabel target:")
display(y.head())

print("\nJumlah fitur:", X.shape[1])
print("Jumlah target:", y.shape[0])

## 9. Membagi data menjadi data latih dan data uji

Data dibagi menjadi 80% data latih dan 20% data uji. Data latih digunakan untuk mempelajari pola, sedangkan data uji digunakan untuk mengevaluasi kemampuan model pada data yang tidak dipakai saat pelatihan.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Jumlah data latih:", len(X_train))
print("Jumlah data uji:", len(X_test))

## 10. Menyiapkan preprocessing

Kolom numerik diproses dengan `StandardScaler`, sementara kolom kategorikal diproses dengan `OneHotEncoder`. `ColumnTransformer` menggabungkan transformasi tersebut dalam satu alur agar diterapkan secara konsisten.

In [ ]:
numeric_features = ["age", "bmi", "children"]
categorical_features = ["sex", "smoker", "region"]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

print("Preprocessing berhasil disiapkan.")

## 11. Melatih dan membandingkan model regresi

Blok ini membuat beberapa model pembanding, termasuk Dummy Regressor, Linear Regression, Decision Tree, Random Forest, dan Gradient Boosting. Setiap model digabungkan dengan preprocessing dalam pipeline lalu dilatih menggunakan `.fit()` pada data latih.

In [ ]:
models = {
    "Dummy Regressor": DummyRegressor(strategy="mean"),
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(
        n_estimators=200,
        random_state=42
    ),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42)
}

trained_models = {}

for name, model in models.items():
    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    # FIT: melatih model dengan data latih
    pipeline.fit(X_train, y_train)

    trained_models[name] = pipeline
    print(f"{name}: selesai dilatih.")

## 12. Mengevaluasi model

Prediksi pada data uji dinilai dengan MAE, RMSE, dan R². MAE dan RMSE mengukur besar kesalahan prediksi (lebih kecil umumnya lebih baik), sedangkan R² mengukur proporsi variasi target yang dijelaskan model (lebih tinggi umumnya lebih baik).

In [ ]:
results = []

for name, model in trained_models.items():
    y_pred = model.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    results.append({
        "Model": name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    })

results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by="RMSE").reset_index(drop=True)

display(results_df.round(4))

## 13. Memilih model terbaik dan memvisualisasikan hasil

Model terbaik dipilih berdasarkan RMSE terkecil pada data uji. Grafik perbandingan RMSE memudahkan melihat perbedaan kinerja antar-model.

In [ ]:
best_model_name = results_df.iloc[0]["Model"]
best_model = trained_models[best_model_name]

print("Model terbaik berdasarkan RMSE:", best_model_name)
print("MAE :", round(results_df.iloc[0]["MAE"], 4))
print("RMSE:", round(results_df.iloc[0]["RMSE"], 4))
print("R²  :", round(results_df.iloc[0]["R2"], 4))

## 14. Validasi silang

Validasi silang 5-fold mengevaluasi model pada beberapa pembagian data latih untuk melihat kestabilan kinerja. Skor yang digunakan adalah RMSE; scikit-learn mengembalikan nilai negatif untuk metrik loss, sehingga tanda dibalik agar hasil RMSE positif.

In [ ]:
plt.figure(figsize=(10, 5))
sns.barplot(data=results_df, x="Model", y="RMSE")
plt.title("Perbandingan RMSE Antar Model")
plt.xlabel("Model")
plt.ylabel("RMSE (lebih rendah lebih baik)")
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()

## 15. Meninjau prediksi pada data uji

Tabel ini membandingkan nilai `charges` aktual dengan hasil prediksi model terbaik dan menghitung selisih absolut. Contoh ini membantu memahami kesalahan prediksi pada tiap sampel.

In [ ]:
cv_scores = cross_val_score(
    best_model,
    X_train,
    y_train,
    cv=5,
    scoring="neg_root_mean_squared_error"
)

cv_rmse = -cv_scores

print("RMSE setiap fold:", np.round(cv_rmse, 2))
print("Rata-rata CV RMSE:", round(cv_rmse.mean(), 2))
print("Standar deviasi CV RMSE:", round(cv_rmse.std(), 2))

## 16. Menyimpan model terbaik

`joblib.dump()` menyimpan pipeline terbaik ke file `insurance_model.joblib`. Karena yang disimpan adalah pipeline, preprocessing dan model ikut tersimpan bersama.

In [ ]:
model_filename = "insurance_model.joblib"

joblib.dump(best_model, model_filename)

print("Model berhasil disimpan sebagai:", model_filename)

## 17. Memuat kembali model dan mencoba prediksi

Kode memuat pipeline yang sudah disimpan, kemudian menggunakannya untuk memprediksi satu baris dari data uji. Ini memeriksa bahwa model tersimpan dapat dimuat kembali dan digunakan.

In [ ]:
loaded_model = joblib.load("insurance_model.joblib")

# Ambil satu baris dari data uji sebagai contoh
contoh_data = X_test.iloc[[0]]
biaya_aktual = y_test.iloc[0]

prediksi = loaded_model.predict(contoh_data)[0]

print("Data contoh:")
display(contoh_data)

print(f"Biaya aktual   : {biaya_aktual:,.2f}")
print(f"Biaya prediksi : {prediksi:,.2f}")
print(f"Selisih absolut: {abs(biaya_aktual - prediksi):,.2f}")

## 18. Kesimpulan otomatis

Bagian akhir merangkum jumlah data setelah pembersihan, model dengan RMSE terkecil, dan metrik evaluasinya. Kesimpulan ini dihasilkan dari nilai yang dihitung oleh notebook, bukan angka yang ditulis secara manual.

In [ ]:
best_result = results_df.iloc[0]

print("KESIMPULAN EVALUASI MODEL")
print("=" * 40)
print(f"Model terbaik berdasarkan RMSE: {best_result['Model']}")
print(f"MAE  : {best_result['MAE']:.2f}")
print(f"RMSE : {best_result['RMSE']:.2f}")
print(f"R²   : {best_result['R2']:.4f}")

if best_result["R2"] >= 0:
    print(
        f"Model menjelaskan sekitar "
        f"{best_result['R2'] * 100:.2f}% variasi biaya asuransi "
        "pada data uji."
    )
else:
    print(
        "Nilai R² negatif: model lebih buruk daripada "
        "prediksi sederhana menggunakan rata-rata target pada data uji."
    )

print("Model telah disimpan ke insurance_model.joblib.")